# 11 — Audit 2024 data before extending training (optional)

**Question:** are earlier PriceCatcher files accessible and sufficiently compatible to justify a longer-history experiment?

Select your project kernel, then **Run All**. This audit does not run 01–10, change configuration, create modelling rows, train models or access the prepared test set. It requires no extra packages. It can run without a completed experiment; cached lookup snapshots, when available, support reference comparisons only.

Check all 12 official monthly URLs using small HTTP requests and Parquet metadata. Sample first/middle/last row groups in **January, July and December only**. Network payloads are capped at 8 MiB per sampled month and 1 MiB per other month (33 MiB maximum, excluding headers). The code refuses a server's full-file fallback. No full monthly files or sampled raw records are saved.

**Scope:** availability, core schema, row counts, sampled dates/prices/codes, occurrence of the current basket and overlap with current shop lookups. A selected-row-group sample is not random or representative of the full year. Zero sampled records does not mean an item was absent. Current lookup units/geography do not establish historical units/geography. Historical monitoring consistency, complete geographical coverage and model improvement remain unassessed.

A cloud-proxy 403 means access is blocked here, not that 2024 data does not exist. The audit records blocked/partial/unknown checks explicitly and stops further requests after a 403. Run on your Mac if its official-source access works.

In [1]:
import io
import json
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from importlib import reload
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pricecatcher_config as cfg
cfg = reload(cfg)
MONTHS = pd.period_range("2024-01", "2024-12", freq="M").astype(str).tolist()
SAMPLE_MONTHS = {"2024-01", "2024-07", "2024-12"}
MIB = 1024 * 1024
CORE = ["date", "premise_code", "item_code", "price"]
OUTPUT_DIR = cfg.DATA_HOME / "11_historical_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class RangeFile(io.RawIOBase):
    """Seekable HTTP reader for Arrow; limits bytes and refuses full-file downloads."""
    def __init__(self, url, limit):
        super().__init__()
        self.url, self.limit = url, limit
        self.position, self.downloaded = 0, 0
        self.cache = {}
        with urlopen(Request(url, method="HEAD", headers={"Accept-Encoding": "identity"}), timeout=20) as response:
            self.size = int(response.headers["Content-Length"])
            self.etag = response.headers.get("ETag")
        if self.size < 12:
            raise ValueError("Object is too small to be a Parquet file.")
    def readable(self):
        return True
    def seekable(self):
        return True
    def tell(self):
        return self.position
    def seek(self, offset, whence=0):
        self.position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset
        if self.position < 0:
            raise ValueError("Negative seek.")
        return self.position
    def read(self, size=-1):
        size = max(0, min(self.size - self.position, self.size if size < 0 else size))
        if not size:
            return b""
        key = (self.position, size)
        if key not in self.cache:
            if self.downloaded + size > self.limit:
                raise ValueError("Audit byte budget reached; remaining checks are unassessed.")
            end = self.position + size - 1
            headers = {"Range": f"bytes={self.position}-{end}", "Accept-Encoding": "identity"}
            if self.etag:
                headers["If-Match"] = self.etag
            with urlopen(Request(self.url, headers=headers), timeout=20) as response:
                expected = f"bytes {self.position}-{end}/{self.size}"
                if response.status != 206 or response.headers.get("Content-Range") != expected:
                    raise ValueError("Exact HTTP range response required; full-file fallback refused.")
                if self.etag and response.headers.get("ETag", self.etag) != self.etag:
                    raise ValueError("Remote object changed during the audit.")
                content = response.read(size)
                self.downloaded += len(content)
                if len(content) != size:
                    raise ValueError("Incomplete range response.")
                self.cache[key] = content
        content = self.cache[key]
        self.position += size
        return content
    def readinto(self, buffer):
        content = self.read(len(buffer))
        buffer[:len(content)] = content
        return len(content)

lookup_status = "unavailable; current reference unit/geography not assessed"
item_reference, premise_reference = None, None
if (cfg.STAGE1_DIR / "source_manifest.json").exists():
    cfg.verify_source_snapshot()
    item_reference = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_item.parquet")
    premise_reference = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_premise.parquet")
    lookup_status = "verified current-study lookup snapshot; not historical evidence"
print("Audit outputs:", OUTPUT_DIR)
print("Reference lookups:", lookup_status)
print("Maximum payload: 33 MiB; sampled row groups are not a full-year coverage assessment.")

Audit outputs: /Users/alfredyap/Documents/GitHub/FYP/data/11_historical_audit
Reference lookups: verified current-study lookup snapshot; not historical evidence
Maximum payload: 33 MiB; sampled row groups are not a full-year coverage assessment.


## Inspect metadata and bounded samples

Each month is assessed independently. Missing files, access failures, budget limits and incompatible schemas are reported rather than treated as passing checks. Dates and codes are parsed in the sample; prices are checked for finite positive values. These checks do not detect every suspicious but positive price.

For sampled months, item counts apply only to the successfully read row groups. The number of selected versus completed row groups is exported. Current-premise lookup overlap measures whether today's reference can join those sampled records; it cannot establish the correctness of historical shop descriptions.

In [2]:
monthly_rows, item_rows = [], []
blocked_reason = None
for month in MONTHS:
    url = cfg.BASE_PRICE_URL.format(month=month)
    row = {"month": month, "url": url, "availability": "not_checked", "metadata_status": "not_assessed",
        "schema_compatible": None, "sample_status": "not_requested", "bytes_read": 0,
        "file_rows": None, "row_groups": None, "requested_sample_groups": 0,
        "completed_sample_groups": 0, "sample_rows": 0, "detail": ""}
    if blocked_reason:
        row["detail"] = "Not requested after access block: " + blocked_reason
        monthly_rows.append(row)
        continue
    reader, samples = None, []
    try:
        reader = RangeFile(url, (8 if month in SAMPLE_MONTHS else 1) * MIB)
        row["availability"] = "available"
        row["file_bytes"] = reader.size
        row["etag"] = reader.etag
        parquet = pq.ParquetFile(reader, pre_buffer=False)
        schema = parquet.schema_arrow
        row["metadata_status"] = "read"
        row["schema"] = str(schema)
        row["file_rows"] = parquet.metadata.num_rows
        row["row_groups"] = parquet.metadata.num_row_groups
        missing = set(CORE) - set(schema.names)
        if missing:
            row["schema_compatible"] = False
            raise ValueError("Missing core columns: " + ", ".join(sorted(missing)))
        date_type = schema.field("date").type
        date_ok = pa.types.is_date(date_type) or pa.types.is_timestamp(date_type) or pa.types.is_string(date_type) or pa.types.is_large_string(date_type)
        numeric_ok = all(pa.types.is_integer(schema.field(c).type) or pa.types.is_floating(schema.field(c).type)
            or pa.types.is_decimal(schema.field(c).type) for c in CORE[1:])
        row["schema_compatible"] = bool(date_ok and numeric_ok)
        if not row["schema_compatible"]:
            raise ValueError("Core column types need explicit compatibility review.")
        if month in SAMPLE_MONTHS:
            group_ids = sorted({0, parquet.metadata.num_row_groups // 2, parquet.metadata.num_row_groups - 1})
            group_ids = [i for i in group_ids if i >= 0]
            row["requested_sample_groups"] = len(group_ids)
            row["sample_status"] = "not_assessed"
            for group_id in group_ids:
                table = parquet.read_row_group(group_id, columns=CORE, use_threads=False)
                samples.append(table.to_pandas())
                row["completed_sample_groups"] += 1
            row["sample_status"] = "sampled" if samples else "empty"
    except Exception as error:
        # Preserve diagnostics; failed network or Arrow checks must not look like passes.
        row["detail"] = f"{type(error).__name__}: {error}"
        if isinstance(error, HTTPError) and error.code == 404:
            row["availability"] = "not_found"
        elif (isinstance(error, HTTPError) and error.code == 403) or "403" in str(error):
            row["availability"] = "access_blocked"
            blocked_reason = row["detail"]
        elif row["availability"] == "not_checked":
            row["availability"] = "request_failed"
        if samples:
            row["sample_status"] = "partial"
    finally:
        if reader is not None:
            row["bytes_read"] = reader.downloaded
            reader.close()
    if samples:
        sample = pd.concat(samples, ignore_index=True)
        dates = pd.to_datetime(sample["date"], errors="coerce")
        prices = pd.to_numeric(sample["price"], errors="coerce")
        codes = pd.to_numeric(sample["item_code"], errors="coerce")
        premises = pd.to_numeric(sample["premise_code"], errors="coerce")
        row["sample_rows"] = len(sample)
        row["sample_date_min"] = str(dates.min())
        row["sample_date_max"] = str(dates.max())
        row["sample_bad_dates"] = int((dates.isna() | dates.dt.to_period("M").ne(pd.Period(month))).sum())
        row["sample_invalid_prices"] = int((prices.isna() | ~np.isfinite(prices) | prices.le(0)).sum())
        row["sample_invalid_codes"] = int((codes.isna() | ~np.isfinite(codes) | codes.mod(1).ne(0)
            | premises.isna() | ~np.isfinite(premises) | premises.mod(1).ne(0)).sum())
        scope = codes.isin(cfg.SCOPE_ITEM_CODES)
        row["sample_basket_rows"] = int(scope.sum())
        row["sample_duplicate_keys"] = int(pd.DataFrame({"date": dates, "premise_code": premises,
            "item_code": codes}).duplicated().sum())
        if premise_reference is not None:
            row["sample_basket_rows_unmatched_current_premises"] = int((scope & ~premises.isin(premise_reference.premise_code)).sum())
        for item_code, group in cfg.PROJECT_ITEM_MAP.items():
            selected = codes.eq(item_code)
            item_row = {"month": month, "item_code": item_code, "project_group": group,
                "sample_records": int(selected.sum()), "sample_premises": int(premises[selected].nunique()),
                "sample_observed_days": int(dates[selected].dt.normalize().nunique()),
                "sample_status": row["sample_status"], "historical_unit_verified": False}
            if item_reference is not None:
                reference = item_reference.loc[item_reference.item_code.eq(item_code)]
                item_row["current_reference_unit"] = reference.unit.iloc[0] if len(reference) == 1 else None
                item_row["current_reference_item"] = reference.item.iloc[0] if len(reference) == 1 else None
            item_rows.append(item_row)
    monthly_rows.append(row)
    print(month, row["availability"], "| metadata:", row["metadata_status"],
        "| sample:", row["sample_status"], "| bytes:", row["bytes_read"])
monthly = pd.DataFrame(monthly_rows)
items = pd.DataFrame(item_rows, columns=["month", "item_code", "project_group", "sample_records",
    "sample_premises", "sample_observed_days", "sample_status", "historical_unit_verified",
    "current_reference_unit", "current_reference_item"])
monthly.to_csv(OUTPUT_DIR / "monthly_availability.csv", index=False)
items.to_csv(OUTPUT_DIR / "selected_item_sample.csv", index=False)
summary = {"checked_at_utc": datetime.now(timezone.utc).isoformat(), "year": 2024,
    "available_months": int(monthly.availability.eq("available").sum()),
    "compatible_metadata_months": int(monthly.schema_compatible.eq(True).sum()),
    "months_with_samples": int(monthly.sample_rows.gt(0).sum()),
    "access_blocked": bool(blocked_reason), "block_reason": blocked_reason,
    "bytes_read": int(monthly.bytes_read.sum()), "lookup_status": lookup_status,
    "historical_units_verified": False, "full_year_coverage_assessed": False,
    "models_or_main_inputs_changed": False,
    "next_step": "Review availability/schema/sample evidence before any full-history experiment; unknown is not passing."}
(OUTPUT_DIR / "audit_summary.json").write_text(json.dumps(summary, indent=2))
display(monthly[["month", "availability", "metadata_status", "schema_compatible", "sample_status",
    "file_rows", "sample_rows", "bytes_read", "detail"]])
if not items.empty:
    display(items.groupby(["item_code", "project_group"])["sample_records"].sum().to_frame())
print(json.dumps(summary, indent=2))
print("No training scope, prepared data, model or prior result was changed.")

2024-01 available | metadata: read | sample: sampled | bytes: 3648214
2024-02 available | metadata: read | sample: not_requested | bytes: 65536
2024-03 available | metadata: read | sample: not_requested | bytes: 65536
2024-04 available | metadata: read | sample: not_requested | bytes: 65536
2024-05 available | metadata: read | sample: not_requested | bytes: 65536
2024-06 available | metadata: read | sample: not_requested | bytes: 65536
2024-07 available | metadata: read | sample: sampled | bytes: 3529751
2024-08 available | metadata: read | sample: not_requested | bytes: 65536
2024-09 available | metadata: read | sample: not_requested | bytes: 65536
2024-10 available | metadata: read | sample: not_requested | bytes: 65536
2024-11 available | metadata: read | sample: not_requested | bytes: 65536
2024-12 available | metadata: read | sample: sampled | bytes: 3684168


,month,availability,metadata_status,schema_compatible,sample_status,file_rows,sample_rows,bytes_read,detail
0,2024-01,available,read,True,sampled,2664434,2664434,3648214,
1,2024-02,available,read,True,not_requested,2083155,0,65536,
2,2024-03,available,read,True,not_requested,2155770,0,65536,
3,2024-04,available,read,True,not_requested,2432981,0,65536,
4,2024-05,available,read,True,not_requested,2593152,0,65536,
5,2024-06,available,read,True,not_requested,2223144,0,65536,
6,2024-07,available,read,True,sampled,2606465,2606465,3529751,
7,2024-08,available,read,True,not_requested,2110317,0,65536,
8,2024-09,available,read,True,not_requested,2702017,0,65536,
9,2024-10,available,read,True,not_requested,2640624,0,65536,


,,sample_records
item_code,project_group,
1,Chicken,62795
2,Chicken,16703
92,Vegetables,60049
93,Vegetables,32306
94,Vegetables,46908
113,Vegetables,81376
114,Vegetables,77027
118,Eggs,7769
119,Eggs,9846


{
  "checked_at_utc": "2026-10-19T16:27:53.404691+00:00",
  "year": 2024,
  "available_months": 12,
  "compatible_metadata_months": 12,
  "months_with_samples": 3,
  "access_blocked": false,
  "block_reason": null,
  "bytes_read": 11451957,
  "lookup_status": "verified current-study lookup snapshot; not historical evidence",
  "historical_units_verified": false,
  "full_year_coverage_assessed": false,
  "models_or_main_inputs_changed": false,
  "next_step": "Review availability/schema/sample evidence before any full-history experiment; unknown is not passing."
}
No training scope, prepared data, model or prior result was changed.


## Decide what the evidence supports

- **Blocked:** run this notebook where official access works. A 403 does not establish missing files.
- **Available with compatible metadata:** the files exist and their core types look usable, not that every record is valid.
- **Sparse/partial samples:** inspect completed sample groups and byte-budget diagnostics. Do not infer full-year absence from a zero count.
- **Current-reference mismatch:** investigate whether historical shops/items can be mapped defensibly. Do not silently discard historical records or claim historical units from today's lookup.
- **Promising audit:** plan a separate 2024–2025 versus 2025-only training comparison. Preserve the 10% target, features, model settings and evaluation population; choose cleaning/item eligibility from each relevant training period. Adding history can change eligibility, so report common-population results and coverage separately.

No model improvement has been tested here. The previously viewed evaluation periods remain exposed. Freeze a proposed longer-history policy before opening a genuinely untouched later outcome period.

Share `monthly_availability.csv`, `selected_item_sample.csv` and `audit_summary.json`. There is no need to upload full monthly files or rerun existing experiments.